# Advanced PySpark for Data Engineering
## Performance, Complex Data, Delta Lake & Production Patterns

**What you'll learn:**
- Advanced window functions in PySpark (frames, ranking, running calculations)
- Complex types: arrays, maps, structs (explode, collect_list, transform)
- UDFs vs built-in functions: when to use each and performance implications
- Advanced joins: broadcast, skew handling, salting technique
- Performance tuning: AQE, caching, checkpointing, repartitioning strategies
- Delta Lake operations: MERGE, time travel, OPTIMIZE, VACUUM, CDC
- Advanced aggregations: cube, rollup, grouping sets
- Spark SQL vs DataFrame API: when to use which
- Writing data: partitioning, bucketing, file compaction
- Production pipeline patterns with error handling

**Prerequisites:** 04_pyspark_deep_dive.ipynb (must have techmart_dw and healthcare_dw created)
**Estimated Time:** 8-10 hours

---

> This notebook takes you from "I can write PySpark" to
> "I can write FAST, RELIABLE PySpark that handles production scale."

---

---
# Section 1: Advanced Window Functions in PySpark

## Beyond Basic Windows: Frames, Complex Rankings, and Analytics

The foundations notebook covered basic window functions.
Here we cover: custom frame specifications, complex multi-window queries,
and production analytics patterns.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Setup
spark.sql("USE techmart_dw")
orders = spark.table("techmart_dw.orders")
customers = spark.table("techmart_dw.customers")
order_items = spark.table("techmart_dw.order_items")
products = spark.table("techmart_dw.products")

print("ADVANCED WINDOW FUNCTIONS")
print("=" * 60)
print(f"  Orders: {orders.count():,} rows")
print(f"  Customers: {customers.count():,} rows")

In [ ]:
# Advanced Window: Multiple windows with custom frames
# Customer purchase behavior analytics

# Define windows
customer_window = Window.partitionBy("customer_id").orderBy("order_date")
customer_window_all = Window.partitionBy("customer_id")
rolling_7 = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(-6, 0)
rolling_30 = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(-29, 0)

customer_analytics = (
    orders
    .filter(col("status") == "completed")
    .select(
        "customer_id", "order_id", "order_date", "total_amount",
        # Running total
        sum("total_amount").over(customer_window).alias("running_total"),
        # Order number for this customer
        row_number().over(customer_window).alias("order_number"),
        # Days since previous order
        datediff(
            col("order_date"),
            lag("order_date").over(customer_window)
        ).alias("days_since_prev"),
        # 7-order moving average
        round(avg("total_amount").over(rolling_7), 2).alias("moving_avg_7"),
        # Percentile within customer's own orders
        round(
            percent_rank().over(Window.partitionBy("customer_id").orderBy("total_amount")),
            3
        ).alias("pct_rank_amount"),
        # Compared to customer average
        round(
            col("total_amount") - avg("total_amount").over(customer_window_all),
            2
        ).alias("diff_from_avg"),
    )
    .filter(col("customer_id").between(1, 5))
    .orderBy("customer_id", "order_date")
)

print("Customer Purchase Analytics (multi-window):")
customer_analytics.show(20, truncate=False)

In [ ]:
# Window: Sessionization in PySpark
# Detect order "sessions" -- groups of orders within 30 days of each other

session_window = Window.partitionBy("customer_id").orderBy("order_date")

sessions = (
    orders
    .filter(col("status") == "completed")
    .withColumn("prev_order_date", lag("order_date").over(session_window))
    .withColumn("days_gap", datediff(col("order_date"), col("prev_order_date")))
    .withColumn("is_new_session",
        when(col("days_gap").isNull() | (col("days_gap") > 30), lit(1)).otherwise(lit(0))
    )
    .withColumn("session_id",
        sum("is_new_session").over(session_window)
    )
)

# Aggregate by session
session_summary = (
    sessions
    .groupBy("customer_id", "session_id")
    .agg(
        count("*").alias("orders_in_session"),
        min("order_date").alias("session_start"),
        max("order_date").alias("session_end"),
        round(sum("total_amount"), 2).alias("session_revenue"),
        datediff(max("order_date"), min("order_date")).alias("session_duration_days")
    )
    .filter(col("customer_id").between(1, 10))
    .orderBy("customer_id", "session_start")
)

print("Sessionization (30-day gap threshold):")
session_summary.show(20, truncate=False)

---
# Section 2: Complex Types -- Arrays, Maps, and Structs

## Handling Nested and Semi-Structured Data

Real-world data is rarely flat. JSON payloads, repeated fields, nested objects --
PySpark has powerful tools for all of them.

| Type | Description | Common Operations |
|------|-------------|-------------------|
| **ArrayType** | List of values | explode, collect_list, array_contains, transform |
| **MapType** | Key-value pairs | map_keys, map_values, element_at |
| **StructType** | Named fields (object) | Dot notation access, struct() |

In [ ]:
# Working with Arrays
print("COMPLEX TYPES: Arrays")
print("=" * 60)

# Collect customer's order amounts into an array
customer_orders = (
    orders
    .filter(col("status") == "completed")
    .groupBy("customer_id")
    .agg(
        collect_list("total_amount").alias("order_amounts"),
        collect_set("payment_method").alias("payment_methods_used"),
        count("*").alias("order_count")
    )
    .filter(col("order_count").between(3, 8))
    .limit(10)
)

print("Arrays from collect_list / collect_set:")
customer_orders.show(5, truncate=False)

# Array operations
array_ops = (
    customer_orders
    .withColumn("num_orders", size("order_amounts"))
    .withColumn("max_order", array_max("order_amounts"))
    .withColumn("min_order", array_min("order_amounts"))
    .withColumn("num_payment_methods", size("payment_methods_used"))
    .withColumn("uses_crypto", array_contains("payment_methods_used", "crypto"))
    .withColumn("sorted_amounts", sort_array("order_amounts", asc=False))
)

print("\nArray operations (size, max, min, contains, sort):")
array_ops.select("customer_id", "num_orders", "max_order", "min_order",
                 "num_payment_methods", "uses_crypto").show(5)

In [ ]:
# Explode: Flatten arrays into rows (inverse of collect_list)
print("\nEXPLODE: Array -> Rows")
print("=" * 60)

# Explode payment methods to analyze each separately
exploded = (
    customer_orders
    .select("customer_id", "order_count", explode("payment_methods_used").alias("payment_method"))
)
print("Exploded payment methods:")
exploded.show(10)

# posexplode: includes the index position
pos_exploded = (
    customer_orders
    .select("customer_id", posexplode("order_amounts").alias("order_index", "amount"))
    .filter(col("customer_id") == customer_orders.first()["customer_id"])
)
print("\nPosexplode (with index):")
pos_exploded.show(10)

In [ ]:
# Working with Maps and JSON
print("\nCOMPLEX TYPES: Maps and JSON Parsing")
print("=" * 60)

# Parse JSON metadata column from customers table
json_parsed = (
    customers
    .filter(col("metadata").isNotNull())
    .withColumn("parsed", from_json(col("metadata"),
        "source STRING, loyalty_tier INT"))
    .withColumn("source", col("parsed.source"))
    .withColumn("loyalty_tier", col("parsed.loyalty_tier"))
    .select("customer_id", "first_name", "metadata", "source", "loyalty_tier")
)
print("JSON parsing with from_json:")
json_parsed.show(10, truncate=False)

# Create a map column
source_summary = (
    json_parsed
    .groupBy("source")
    .agg(
        count("*").alias("customer_count"),
        round(avg("loyalty_tier"), 2).alias("avg_tier")
    )
)
print("\nCustomer acquisition by source:")
source_summary.show()

In [ ]:
# Higher-order functions: transform, filter, aggregate on arrays
print("\nHIGHER-ORDER FUNCTIONS on Arrays")
print("=" * 60)

# transform: apply function to each element
# filter: keep elements matching condition
# aggregate: reduce array to single value

array_demo = (
    customer_orders
    .withColumn("amounts_doubled",
        transform("order_amounts", lambda x: x * 2))
    .withColumn("high_value_orders",
        filter("order_amounts", lambda x: x > 2000))
    .withColumn("total_from_array",
        aggregate("order_amounts", lit(0.0), lambda acc, x: acc + x))
    .select("customer_id", "order_amounts", "amounts_doubled",
            "high_value_orders", "total_from_array")
)

print("transform, filter, aggregate on arrays:")
array_demo.show(5, truncate=False)

---
# Section 3: UDFs vs Built-in Functions

## The #1 Performance Mistake in PySpark

**Rule:** ALWAYS prefer built-in functions over UDFs.

| Approach | Speed | Why |
|----------|-------|-----|
| Built-in (col, when, etc.) | Fastest | Runs in JVM, optimized by Catalyst |
| Pandas UDF (vectorized) | ~10x slower | Batched Arrow transfer, vectorized |
| Python UDF (row-by-row) | ~100x slower | Serializes each row Python<->JVM |

**When to use UDFs:**
- Complex logic that CAN'T be expressed with built-in functions
- Calling external libraries (ML models, custom parsers)
- Business logic too complex for CASE WHEN chains

In [ ]:
# Performance comparison: Built-in vs UDF
import time
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd

# The task: categorize orders by amount
# APPROACH 1: Built-in functions (FAST)
def builtin_categorize(df):
    return df.withColumn("tier",
        when(col("total_amount") >= 4000, "platinum")
        .when(col("total_amount") >= 2000, "gold")
        .when(col("total_amount") >= 1000, "silver")
        .otherwise("bronze")
    )

# APPROACH 2: Python UDF (SLOW)
@udf(returnType=StringType())
def categorize_udf(amount):
    if amount is None:
        return "unknown"
    if amount >= 4000:
        return "platinum"
    elif amount >= 2000:
        return "gold"
    elif amount >= 1000:
        return "silver"
    return "bronze"

def udf_categorize(df):
    return df.withColumn("tier", categorize_udf(col("total_amount")))

# APPROACH 3: Pandas UDF (MEDIUM -- vectorized)
@pandas_udf(StringType())
def categorize_pandas_udf(amounts: pd.Series) -> pd.Series:
    conditions = [
        amounts >= 4000,
        amounts >= 2000,
        amounts >= 1000,
    ]
    choices = ["platinum", "gold", "silver"]
    import numpy as np
    return pd.Series(np.select(conditions, choices, default="bronze"))

def pandas_udf_categorize(df):
    return df.withColumn("tier", categorize_pandas_udf(col("total_amount")))

# Benchmark
print("UDF PERFORMANCE COMPARISON")
print("=" * 60)

df = orders.select("order_id", "total_amount")

for name, func in [("Built-in", builtin_categorize), ("Pandas UDF", pandas_udf_categorize), ("Python UDF", udf_categorize)]:
    start = time.perf_counter()
    result = func(df)
    result.count()  # Force execution
    elapsed = time.perf_counter() - start
    print(f"  {name:<12}: {elapsed:.3f}s")

print("\n  Takeaway: Built-in is ALWAYS fastest. Use UDFs only when necessary.")
print("  If you must use a UDF, prefer Pandas UDF (vectorized) over Python UDF.")

---
# Section 4: Advanced Joins & Skew Handling

## Broadcast Joins, Skew Detection, and Salting

Join performance is the #1 optimization opportunity in Spark.
Understanding join strategies saves HOURS of runtime on large datasets.

In [ ]:
# Broadcast join: Force small table into memory on all executors
from pyspark.sql.functions import broadcast

print("ADVANCED JOINS")
print("=" * 60)

# Check table sizes
print("Table sizes:")
for name, df in [("orders", orders), ("products", products), ("customers", customers)]:
    count = df.count()
    print(f"  {name}: {count:,} rows")

# Broadcast join (small table broadcast to all executors)
# Products (500 rows) should ALWAYS be broadcast
start = time.perf_counter()
broadcast_result = (
    order_items
    .join(broadcast(products), "product_id")
    .groupBy("category")
    .agg(
        count("*").alias("items_sold"),
        round(sum("line_total"), 2).alias("revenue")
    )
)
broadcast_result.count()
elapsed = time.perf_counter() - start
print(f"\n  Broadcast join (products): {elapsed:.3f}s")

# Show the join strategy in the plan
print("\nExplain (look for BroadcastHashJoin):")
broadcast_result.explain(mode="simple")

In [ ]:
# Skew Handling: The Salting Technique
# Problem: Some customer_ids have 1000x more orders than others (data skew)
# Solution: Add a random "salt" to distribute hot keys across partitions

print("\nSKEW HANDLING: Salting Technique")
print("=" * 60)

# Detect skew: find hot keys
order_distribution = (
    orders
    .groupBy("customer_id")
    .count()
    .orderBy(col("count").desc())
)
print("Top 5 customers by order count (potential skew):")
order_distribution.show(5)

# Salting technique for skewed joins
num_salts = 10

# Step 1: Add salt to the large (skewed) table
orders_salted = orders.withColumn("salt", (rand() * num_salts).cast("int"))

# Step 2: Explode the small table with all salt values
customers_exploded = (
    customers
    .crossJoin(
        spark.range(num_salts).withColumnRenamed("id", "salt")
    )
)

# Step 3: Join on composite key (original key + salt)
salted_join = (
    orders_salted
    .join(
        customers_exploded,
        (orders_salted.customer_id == customers_exploded.customer_id) &
        (orders_salted.salt == customers_exploded.salt)
    )
    .select(orders_salted["order_id"], orders_salted["total_amount"],
            customers_exploded["first_name"], customers_exploded["customer_segment"])
)

print(f"\n  Salted join result: {salted_join.count():,} rows")
print("  Skew is now distributed across 10 salt values per hot key!")
print("  Use this when you see long-running tasks in Spark UI (straggler tasks).")

---
# Section 5: Performance Tuning

## AQE, Caching, Partitioning Strategies & Explain Plans

These are the knobs you turn to make Spark jobs run 10-100x faster.

In [ ]:
# Adaptive Query Execution (AQE) -- enabled by default in modern Spark
print("PERFORMANCE TUNING")
print("=" * 60)

# Check AQE status
aqe_enabled = spark.conf.get("spark.sql.adaptive.enabled", "unknown")
print(f"  AQE enabled: {aqe_enabled}")
print("  AQE automatically:")
print("    - Coalesces post-shuffle partitions (reduces small files)")
print("    - Converts sort-merge joins to broadcast when one side is small")
print("    - Handles skew by splitting large partitions")

# Partition management
print(f"\n  Default shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

# For our dataset size, 200 partitions is too many. Reduce for small data:
spark.conf.set("spark.sql.shuffle.partitions", "8")
print(f"  Set to 8 for our dataset size (< 1GB)")

In [ ]:
# Caching: When and how to cache DataFrames
print("\nCACHING STRATEGIES")
print("=" * 60)

# Cache when: DataFrame is reused multiple times
# DON'T cache when: used only once, or too large for memory

# Example: orders used in multiple aggregations
orders_completed = orders.filter(col("status") == "completed").cache()

# First action materializes the cache
start = time.perf_counter()
count1 = orders_completed.count()
t1 = time.perf_counter() - start

# Second action uses cached data (much faster)
start = time.perf_counter()
count2 = orders_completed.groupBy("payment_method").count().count()
t2 = time.perf_counter() - start

# Third use
start = time.perf_counter()
count3 = orders_completed.agg(sum("total_amount")).collect()
t3 = time.perf_counter() - start

print(f"  First use (cache materialization): {t1:.3f}s")
print(f"  Second use (from cache): {t2:.3f}s")
print(f"  Third use (from cache): {t3:.3f}s")

# IMPORTANT: Unpersist when done!
orders_completed.unpersist()
print("\n  Always unpersist() when done to free memory!")
print("  cache() = MEMORY_AND_DISK (default)")
print("  persist(StorageLevel.MEMORY_ONLY) = faster but may OOM")
print("  persist(StorageLevel.DISK_ONLY) = safest for large DataFrames")

In [ ]:
# Reading execution plans -- what to look for
print("\nREADING EXPLAIN PLANS")
print("=" * 60)

# Complex query
complex_query = (
    orders
    .filter(col("status") == "completed")
    .filter(col("order_date") >= "2023-01-01")
    .join(broadcast(products.select("product_id", "category")),
          order_items.select("order_id", "product_id", "line_total"),
          "product_id")
    .groupBy("category")
    .agg(sum("line_total").alias("revenue"))
    .orderBy(col("revenue").desc())
)

print("Query plan analysis:")
print("Look for:")
print("  - FileScan (what's being read, pushdown predicates)")
print("  - BroadcastHashJoin vs SortMergeJoin")
print("  - Exchange (shuffle) -- fewer is better")
print("  - Project (column pruning)")
print()
complex_query.explain(mode="simple")

---
# Section 6: Delta Lake Advanced Operations

## MERGE, Time Travel, OPTIMIZE, VACUUM, and CDC

Delta Lake is the foundation of the Lakehouse. These operations are
what you'll use daily as a Databricks data engineer.

In [ ]:
# Delta Lake: MERGE (upsert) from PySpark
print("DELTA LAKE OPERATIONS")
print("=" * 60)

# Create a Delta table for demonstration
spark.sql("""
    CREATE OR REPLACE TABLE techmart_dw.customer_metrics
    USING DELTA AS
    SELECT
        customer_id,
        COUNT(*) AS total_orders,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        MAX(order_date) AS last_order_date,
        current_timestamp() AS updated_at
    FROM techmart_dw.orders
    WHERE status = 'completed'
    GROUP BY customer_id
""")

print("  Created customer_metrics Delta table")
print(f"  Rows: {spark.table('techmart_dw.customer_metrics').count():,}")

In [ ]:
# MERGE: Upsert new metrics
from delta.tables import DeltaTable

# Simulate new/updated metrics (as if recalculated today)
new_metrics = spark.sql("""
    SELECT
        customer_id,
        COUNT(*) AS total_orders,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        MAX(order_date) AS last_order_date,
        current_timestamp() AS updated_at
    FROM techmart_dw.orders
    WHERE status IN ('completed', 'shipped')
    GROUP BY customer_id
    LIMIT 100
""")

# Perform MERGE
delta_table = DeltaTable.forName(spark, "techmart_dw.customer_metrics")

merge_result = (
    delta_table.alias("target")
    .merge(new_metrics.alias("source"), "target.customer_id = source.customer_id")
    .whenMatchedUpdate(
        condition="source.total_revenue != target.total_revenue",
        set={
            "total_orders": "source.total_orders",
            "total_revenue": "source.total_revenue",
            "last_order_date": "source.last_order_date",
            "updated_at": "source.updated_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("  MERGE completed (upsert 100 customer metrics)")
print(f"  Table now has: {spark.table('techmart_dw.customer_metrics').count():,} rows")

In [ ]:
# Time Travel: Query previous versions
print("\nTIME TRAVEL")
print("=" * 60)

# View table history
history = spark.sql("DESCRIBE HISTORY techmart_dw.customer_metrics")
print("Table history (versions):")
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

# Query a previous version
print("\nVersion 0 (original) row count:")
v0 = spark.read.format("delta").option("versionAsOf", 0).table("techmart_dw.customer_metrics")
print(f"  Version 0: {v0.count():,} rows")

current = spark.table("techmart_dw.customer_metrics")
print(f"  Current:   {current.count():,} rows")

In [ ]:
# OPTIMIZE and VACUUM
print("\nOPTIMIZE & VACUUM")
print("=" * 60)

# OPTIMIZE: Compact small files into larger ones (better read performance)
spark.sql("OPTIMIZE techmart_dw.customer_metrics")
print("  OPTIMIZE completed (compacted small files)")

# VACUUM: Remove old files no longer needed (free storage)
# Default retention: 7 days (to support time travel)
# spark.sql("VACUUM techmart_dw.customer_metrics RETAIN 168 HOURS")
print("  VACUUM removes old Delta log files after retention period")
print("  Default: 7 days (168 hours)")
print("  Production tip: Run VACUUM weekly, OPTIMIZE daily")

# Check file metrics
detail = spark.sql("DESCRIBE DETAIL techmart_dw.customer_metrics")
detail.select("format", "numFiles", "sizeInBytes").show()

---
# Section 7: Writing Data Efficiently

## Partitioning, Coalescing, and File Management

How you WRITE data determines how fast it can be READ downstream.
Bad writes = thousands of tiny files = slow queries for everyone.

In [ ]:
# Writing strategies
print("WRITING DATA EFFICIENTLY")
print("=" * 60)

# Check current partition count before writing
completed_orders = orders.filter(col("status") == "completed")
print(f"  DataFrame partitions: {completed_orders.rdd.getNumPartitions()}")

# Strategy 1: Coalesce before write (reduce small files)
print("\n  Strategy 1: coalesce(4) -- reduce to 4 files")
(completed_orders
    .coalesce(4)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("techmart_dw.orders_optimized_v1")
)
files_v1 = spark.sql("DESCRIBE DETAIL techmart_dw.orders_optimized_v1").select("numFiles").first()[0]
print(f"  Files written: {files_v1}")

# Strategy 2: Partition by date (for time-based queries)
print("\n  Strategy 2: partitionBy('partition_date') -- partition by month")
(completed_orders
    .repartition("partition_date")
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("partition_date")
    .saveAsTable("techmart_dw.orders_optimized_v2")
)
files_v2 = spark.sql("DESCRIBE DETAIL techmart_dw.orders_optimized_v2").select("numFiles").first()[0]
print(f"  Files written: {files_v2} (one per partition)")
print("  Benefit: queries filtering on partition_date only read relevant files!")

# Clean up
spark.sql("DROP TABLE IF EXISTS techmart_dw.orders_optimized_v1")
spark.sql("DROP TABLE IF EXISTS techmart_dw.orders_optimized_v2")
print("\n  Cleaned up demo tables")

---
# Section 8: Advanced Aggregations

## Cube, Rollup, and Grouping Sets in PySpark

In [ ]:
# CUBE: All dimension combinations
print("ADVANCED AGGREGATIONS: Cube & Rollup")
print("=" * 60)

# Rollup: hierarchical subtotals
rollup_result = (
    orders
    .filter(col("status") == "completed")
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .rollup("year", "month", "payment_method")
    .agg(
        count("*").alias("order_count"),
        round(sum("total_amount"), 2).alias("revenue")
    )
    .orderBy("year", "month", "payment_method")
    .filter(col("year").isNotNull())  # Remove grand total for display
)

print("ROLLUP (year > month > payment_method with subtotals):")
rollup_result.show(20)

# Cube: all possible dimension combinations
cube_result = (
    orders
    .filter((col("status") == "completed") & (col("order_date") >= "2023-06-01"))
    .cube("order_source", "payment_method")
    .agg(
        count("*").alias("orders"),
        round(sum("total_amount"), 2).alias("revenue")
    )
    .filter(col("orders") > 50)
    .orderBy(col("revenue").desc())
)

print("\nCUBE (all combinations of source x payment_method):")
cube_result.show(15)

---
# Section 9: Your Turn -- Production Challenges

In [ ]:
# ============================================
# YOUR TURN: Challenge 1 -- Customer 360 View
# ============================================
# Build a comprehensive customer profile by combining:
# - customers table (demographics)
# - orders (purchase behavior)
# - order_items + products (product preferences)
#
# Output should include:
# - customer_id, name, segment
# - total_orders, total_revenue, avg_order_value
# - first_order_date, last_order_date, days_since_last_order
# - favorite_category (most purchased)
# - favorite_payment_method
# - order_frequency_days (avg days between orders)
#
# Use window functions, aggregations, and proper join strategies.
# Hint: broadcast(products), use multiple aggregation levels
#
# Write your solution below:
print("Challenge 1: Build a Customer 360 View")
print("Combine demographics + purchase behavior + product preferences")


In [ ]:
# ============================================
# YOUR TURN: Challenge 2 -- Incremental MERGE Pipeline
# ============================================
# Build a production-ready incremental pipeline that:
# 1. Reads "new" orders (simulate with orders from last 30 days)
# 2. Transforms them (add computed columns)
# 3. MERGEs into a target Delta table (upsert by order_id)
# 4. Handles both updates AND new inserts
# 5. Tracks merge metrics (rows inserted, updated)
#
# This is THE most common pattern in production Databricks pipelines.
#
# Write your solution below:
print("Challenge 2: Build an Incremental MERGE Pipeline")
print("Simulate extract -> transform -> merge pattern")


---
# Summary

## Advanced PySpark Cheat Sheet

| Topic | Key Takeaway |
|-------|-------------|
| **Window Frames** | ROWS BETWEEN for physical, RANGE BETWEEN for logical |
| **Complex Types** | explode to flatten, collect_list to nest, transform for element-wise |
| **UDFs** | Built-in > Pandas UDF > Python UDF (100x performance difference) |
| **Broadcast Join** | Use for tables < 10MB (products, lookup tables) |
| **Skew Handling** | Salt hot keys to distribute across partitions |
| **AQE** | Enabled by default -- auto-optimizes joins and partitions |
| **Caching** | Only for reused DataFrames. Always unpersist() after |
| **Delta MERGE** | Production upsert pattern. Conditional updates save writes |
| **Time Travel** | Query any previous version. Retention = 7 days default |
| **OPTIMIZE** | Run daily to compact small files |
| **Partitioning** | Partition by query filter columns (date, region) |
| **Coalesce vs Repartition** | Coalesce to reduce (no shuffle), Repartition to increase |

## Performance Optimization Checklist

1. Filter early (predicate pushdown)
2. Select only needed columns (column pruning)
3. Broadcast small tables (< 10MB)
4. Reduce shuffle partitions for small data
5. Cache reused DataFrames (unpersist after)
6. Avoid Python UDFs (use built-in functions)
7. Check explain() for unexpected shuffles
8. Coalesce before writing (avoid small files)
9. Partition writes by query filter columns
10. OPTIMIZE Delta tables regularly

---
*Advanced PySpark for Data Engineering -- Using techmart_dw datasets*

In [ ]:
print("KEY TAKEAWAYS FROM ADVANCED PYSPARK")
print("=" * 50)
print()
print("1. Window functions: frames, sessionization, running totals")
print("2. Complex types: explode/collect_list, JSON parsing, higher-order fns")
print("3. UDFs: LAST RESORT. Built-in functions are 100x faster")
print("4. Joins: broadcast small tables, salt for skew")
print("5. Performance: AQE, caching, partition tuning, explain()")
print("6. Delta: MERGE for upserts, time travel, OPTIMIZE, VACUUM")
print("7. Writing: coalesce to control file count, partition by filter cols")
print()
print("Production mantra:")
print("  Filter early, select narrow, broadcast small, avoid UDFs,")
print("  check the plan, compact your files.")